# notebooks/04_generation_test.ipynb

In [1]:
%run notebook_init.py

In [2]:
from rag_pipeline.retriever.retrieve import query_chunks
from rag_pipeline.verifier.hallucination_guard import sentence_overlap
from rag_pipeline.llm.generator import generate_answer, format_prompt, generate_answer_with_gate

/Users/aditya/Documents/projects/hallucination-resistant-finance-rag/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
dryrun = True # Set to TRUE by default

In [ ]:
def ask_and_verify():
    while True:
        q = input("Ask a question (or 'q' to quit): ")
        if q.lower() == 'q':
            break
        chunks = query_chunks(q)

        print("\n🔎 Retrieved Chunks (Debug Info):")
        for i, c in enumerate(chunks):
            metadata = c.get('metadata', {})
            print(f"\n  Chunk {i+1}:")
            print(f"    doc_id: {metadata.get('doc_id', 'unknown')}")
            print(f"    chunk_id: {metadata.get('chunk_id', 'unknown')}")
            print(f"    section: {metadata.get('section', 'unknown')}")
            print(f"    distance: {c.get('distance', 'N/A')}")
            print(f"    text preview: {c['text'][:200]}...")

        answer, abstained = generate_answer_with_gate(q, chunks, model_name="gpt-4o")
        print("\nAnswer:\n", answer)
        if abstained:
            print("(System abstained from answering)")

In [ ]:
ask_and_verify()

In [ ]:
# Use query_chunks() as single source of truth for retrieval
# This ensures consistent metadata and avoids mismatched Chroma instances

for i, c in enumerate(chunks):
    print(f"\n--- Chunk {i+1} ---")
    print(f"doc_id: {c['metadata'].get('doc_id', 'unknown')}")
    print(f"chunk_id: {c['metadata'].get('chunk_id', 'unknown')}")
    print(f"section: {c['metadata'].get('section', 'unknown')}")
    print(f"distance: {c.get('distance', 'N/A')}")
    print(f"text preview: {c['text'][:300]}...")

# Debug Code Below

In [ ]:
from rag_pipeline.retriever.retrieve import query_chunks
from rag_pipeline.llm.generator import generate_answer_with_gate

# Test multiple queries
queries = [
    # --- HQ / Location ---
    "Where is Tesla headquartered?",
    "What city is Tesla based in?",
    "Where is Tesla's headquarters located?",

    # --- Incorporation / Legal ---
    # "Where is Tesla incorporated?",
    # "What state is Tesla incorporated in?",
    #"Is Tesla a Delaware corporation?",

    # --- Business Overview ---
    #"What does Tesla do?",
    #Describe Tesla's business model.",
    #"What are Tesla's main lines of business?",

    # --- Financials / Revenue ---
    #"What was Tesla's revenue in 2023?",
    #"How much money did Tesla make last year?",
    #"What were Tesla's total revenues?",

    # --- Risks ---
    #"What are Tesla's major risks?",
    #"What regulatory risks does Tesla face?",
    #"What could negatively impact Tesla's business?",

    # --- Auditor / Accounting ---
    #"Who audits Tesla?",
    #"What accounting firm reviews Tesla's financials?",
    #"Who is Tesla's independent registered public accounting firm?",

    # --- Forced Abstention (should abstain) ---
    #"What are Tesla's plans for nuclear energy?",
    #"What was Tesla's revenue in India?",
    #"Who is Tesla's largest customer in South America?",
    #"What subsidies does Tesla receive from Brazil?",
]

for query in queries:
    print(f"\n{'='*60}")
    print(f"Query: {query}")
    print('='*60)
    
    chunks = query_chunks(query, top_k=5)
    answer, abstained = generate_answer_with_gate(query, chunks)
    print(f"================================\nAnswer: {answer}\n=================================")
    print(f"Abstained: {abstained}")